# Interactive Exploration — Ensemble Methods: Boosting vs. Bagging

Live, slider-driven views of every from-scratch model in this project:

1. **Decision boundaries** — DecisionTree vs. AdaBoost vs. RandomForest side by side on toy 2D datasets.
2. **AdaBoost anatomy** — staged train/test error and per-round estimator weights as the round count grows.
3. **Unsupervised view** — PCA projection of breast cancer with live K-Means (k slider, ARI) and DBSCAN (eps slider, noise fraction).

Run all cells top to bottom, then play with the sliders. Everything uses the `src/` implementations, not sklearn.
Each figure's latest state is also saved as a PNG under `figures/notebook/`.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_circles, make_classification, make_moons
from sklearn.metrics import adjusted_rand_score

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    print("ipywidgets tapilmadi - qrafikler statik rejimde cekilecek")

# Layihə kökünü tapırıq: cwd-dən yuxarı qalxaraq "src" qovluğunu axtarırıq,
# beləcə notebook istər kökdən, istər notebooks/ içindən açılsın — işləyir.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Could not locate the project root (folder containing 'src')")
sys.path.insert(0, str(PROJECT_ROOT))

from src.trees.decision_tree import DecisionTree
from src.boosting.adaboost import AdaBoostClassifier
from src.bagging.random_forest import RandomForestClassifier
from src.unsupervised.pca import PCA
from src.unsupervised.kmeans import KMeans
from src.unsupervised.dbscan import DBSCAN

RANDOM_SEED = 42

# Qrafiklərin son vəziyyəti PNG kimi bura yazılır.
FIGURES_DIR = PROJECT_ROOT / "figures" / "notebook"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def show_and_save(fig, filename: str) -> None:
    """Save the figure under figures/notebook/ and render it inline."""
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


def make_dataset(name: str, n_samples: int = 250):
    if name == "clusters":
        return make_classification(n_samples=n_samples, n_features=2, n_redundant=0,
                                   n_informative=2, random_state=RANDOM_SEED,
                                   n_clusters_per_class=1)
    if name == "moons":
        return make_moons(n_samples=n_samples, noise=0.25, random_state=RANDOM_SEED)
    return make_circles(n_samples=n_samples, noise=0.15, factor=0.5,
                        random_state=RANDOM_SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures dir : {FIGURES_DIR}")

## 1. One dataset, three models

The same 2D dataset is classified by a single tree, AdaBoost over stumps, and a Random Forest.
Watch how the single tree carves axis-aligned rectangles, boosting builds its boundary out of
many weighted depth-1 cuts, and bagging smooths the tree's jagged edges by averaging.

In [ ]:
def plot_three_boundaries(dataset, tree_depth, n_estimators):
    X, y = make_dataset(dataset)
    models = {
        f"DecisionTree (depth={tree_depth})":
            DecisionTree(max_depth=tree_depth, random_state=RANDOM_SEED).fit(X, y),
        f"AdaBoost ({n_estimators} stumps)":
            AdaBoostClassifier(n_estimators=n_estimators,
                               random_state=RANDOM_SEED).fit(X, y),
        f"RandomForest ({n_estimators} trees)":
            RandomForestClassifier(n_estimators=n_estimators, max_depth=tree_depth,
                                   random_state=RANDOM_SEED).fit(X, y),
    }

    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.07),
                         np.arange(y_min, y_max, 0.07))
    grid = np.c_[xx.ravel(), yy.ravel()]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    for axis, (title, model) in zip(axes, models.items()):
        Z = model.predict(grid).reshape(xx.shape)
        train_acc = float(np.mean(model.predict(X) == y))
        axis.contourf(xx, yy, Z, alpha=0.4, cmap="RdBu")
        axis.scatter(X[:, 0], X[:, 1], c=y, s=16, edgecolor="k", cmap="RdBu")
        axis.set_title(title + chr(10) + f"train acc={train_acc:.3f}")
    fig.suptitle(f"Dataset: {dataset}")
    fig.tight_layout()
    show_and_save(fig, f"boundaries_{dataset}.png")


if HAS_WIDGETS:
    boundary_ui = widgets.interactive(
        plot_three_boundaries,
        dataset=widgets.Dropdown(options=["moons", "clusters", "circles"],
                                 value="moons", description="Dataset"),
        tree_depth=widgets.IntSlider(min=1, max=10, step=1, value=3,
                                     description="Tree depth"),
        n_estimators=widgets.IntSlider(min=5, max=60, step=5, value=25,
                                       description="Ensemble size"),
    )
    display(boundary_ui)
else:
    plot_three_boundaries("moons", 3, 25)

## 2. Inside AdaBoost: staged error and estimator weights

`staged_predict` replays the ensemble round by round. The left panel shows train/test error as
rounds accumulate (drag the noise slider to see boosting latch onto mislabeled points); the right
panel shows each round's weight (alpha_m = ln((1-eps_m)/eps_m)) — early accurate stumps earn
large votes, later ones fight over the hard leftovers.

In [ ]:
def plot_adaboost_anatomy(n_estimators, noise):
    X, y = make_dataset("moons", n_samples=400)
    rng = np.random.default_rng(RANDOM_SEED)
    split = rng.permutation(len(y))
    train_idx, test_idx = split[:280], split[280:]
    X_train, y_train = X[train_idx], y[train_idx].copy()
    X_test, y_test = X[test_idx], y[test_idx]

    n_flip = int(noise * len(y_train))
    if n_flip:
        flip = rng.choice(len(y_train), size=n_flip, replace=False)
        y_train[flip] = 1 - y_train[flip]

    model = AdaBoostClassifier(n_estimators=n_estimators,
                               random_state=RANDOM_SEED).fit(X_train, y_train)
    train_err = [np.mean(stage != y_train) for stage in model.staged_predict(X_train)]
    test_err = [np.mean(stage != y_test) for stage in model.staged_predict(X_test)]
    rounds = np.arange(1, len(train_err) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(rounds, train_err, label="Train error")
    axes[0].plot(rounds, test_err, label="Test error")
    axes[0].set(xlabel="Boosting round", ylabel="Error",
                title=f"Staged error (label noise={noise:.0%})")
    axes[0].legend()
    axes[1].bar(rounds, model.estimator_weights, width=0.8)
    axes[1].set(xlabel="Boosting round", ylabel="alpha_m",
                title="Estimator weights")
    for axis in axes:
        axis.grid(alpha=0.25)
    fig.tight_layout()
    show_and_save(fig, "adaboost_anatomy.png")


if HAS_WIDGETS:
    adaboost_ui = widgets.interactive(
        plot_adaboost_anatomy,
        n_estimators=widgets.IntSlider(min=5, max=120, step=5, value=40,
                                       description="Rounds"),
        noise=widgets.FloatSlider(min=0.0, max=0.3, step=0.05, value=0.0,
                                  description="Label noise", readout_format=".0%"),
    )
    display(adaboost_ui)
else:
    plot_adaboost_anatomy(40, 0.0)

## 3. Unsupervised view: PCA + K-Means + DBSCAN on breast cancer

The 30-feature dataset is projected onto its first two principal components (our `PCA`).
The k slider re-runs our `KMeans` (ARI against the true diagnosis shown in the title);
the eps slider re-runs our `DBSCAN` — red crosses are points labeled as noise.

In [ ]:
_cancer = load_breast_cancer()
_X_std = (_cancer.data - _cancer.data.mean(axis=0)) / _cancer.data.std(axis=0)
_projection = PCA(n_components=2).fit_transform(_X_std)


def plot_unsupervised(k, eps):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED).fit(_X_std)
    dbscan = DBSCAN(eps=eps, min_samples=5).fit(_X_std)
    kmeans_ari = adjusted_rand_score(_cancer.target, kmeans.labels_)
    dbscan_ari = adjusted_rand_score(_cancer.target, dbscan.labels_)
    noise_fraction = float(np.mean(dbscan.labels_ == -1))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    axes[0].scatter(_projection[:, 0], _projection[:, 1], c=_cancer.target,
                    cmap="coolwarm", s=14)
    axes[0].set_title("True diagnosis")
    axes[1].scatter(_projection[:, 0], _projection[:, 1], c=kmeans.labels_,
                    cmap="viridis", s=14)
    axes[1].set_title(f"K-Means (k={k}), ARI={kmeans_ari:.3f}")
    clustered = dbscan.labels_ != -1
    axes[2].scatter(_projection[clustered, 0], _projection[clustered, 1],
                    c=dbscan.labels_[clustered], cmap="viridis", s=14)
    axes[2].scatter(_projection[~clustered, 0], _projection[~clustered, 1],
                    marker="x", color="tab:red", s=22)
    axes[2].set_title(f"DBSCAN (eps={eps:.1f}), ARI={dbscan_ari:.3f}, "
                      f"noise={noise_fraction:.0%}")
    for axis in axes:
        axis.set(xlabel="PC1", ylabel="PC2")
        axis.grid(alpha=0.2)
    fig.tight_layout()
    show_and_save(fig, "unsupervised_breast_cancer.png")


if HAS_WIDGETS:
    unsupervised_ui = widgets.interactive(
        plot_unsupervised,
        k=widgets.IntSlider(min=2, max=8, step=1, value=2, description="k"),
        eps=widgets.FloatSlider(min=1.0, max=8.0, step=0.5, value=4.0,
                                description="eps"),
    )
    display(unsupervised_ui)
else:
    plot_unsupervised(2, 4.0)